# Demo: delegation flow of the multi-agent orchestrator

Run this notebook from the project root kernel (`.venv`). It needs `GOOGLE_API_KEY` in `.env`.

Expected flow: **supervisor -> researcher -> supervisor -> analyst -> supervisor -> synthesizer**.

In [1]:
import os, sys
os.chdir(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
from src.graph import build_graph, initial_state
from src.main import DEFAULT_TASK, format_event
graph = build_graph()

## Graph topology

In [2]:
from IPython.display import Markdown
Markdown('```mermaid\n' + graph.get_graph().draw_mermaid() + '\n```')

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	researcher --> supervisor;
	supervisor -.-> analyst;
	supervisor -.-> researcher;
	supervisor -.-> synthesizer;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run and watch each delegation step

In [3]:
contributions, decisions = [], []
for event in graph.stream(initial_state(DEFAULT_TASK), config={'recursion_limit': 30}, stream_mode='updates'):
    for node, update in event.items():
        print(format_event(node, update), '\n')
        contributions += update.get('contributions', [])
        if node == 'supervisor':
            decisions.append(update['next_agent'])

[supervisor] decision 1 -> researcher
    reasoning: Nothing has been done yet. According to the routing rules, we must start with the researcher to gather facts, quotes, and ratings for LangGraph, CrewAI, and AutoGen.
    instructions: Search for community opinions, user reviews, ratings, and discussions regarding LangGraph, CrewAI, and AutoGen for agent orchestration. Collect specific quotes, ratings, and numeric data or sentiment indicators for each framework with their sources. 

[researcher] contributed (decision 1):
Here are the findings from the knowledge base regarding community opinions, user reviews, ratings, and discussions for LangGraph, CrewAI, and AutoGen:

* **LangGraph**
  * **Entity:** LangGraph | **Source:** kb-01 | **Rating:** 5 | **Quote:** "LangGraph gives excellent control over agent flows. Explicit state and conditional edges make the behaviour easy to debug and very reliable."
  * **Entity:** LangGraph | **Source:** kb-02 | **Rating:** 4 | **Quote:** "Muy potent

## Who contributed what (shared state)

In [4]:
for c in contributions:
    print(f"decision {c['step']} | {c['agent']:<10} | {len(c['output'])} chars")
print('supervisor route:', ' -> '.join(decisions))

decision 1 | researcher | 2890 chars
decision 2 | analyst    | 1992 chars
supervisor route: researcher -> analyst -> synthesizer
